In [ ]:
from docopt import docopt
from calculate_top_n import calculate_top_n
import pandas as pd
import mudata
import os
# Custom functions import
from prepare_mogonet_single_split import prepare_mogonet_feat_select
#from feature_importance import  cal_feat_imp, summarize_imp_feat


In [2]:
method = "mogonet"
mu_path = "../../../../../../tenFoldCV_results/prepare_data/prepare_mu_data/breast_tcga/breast_tcga.h5mu"
dataset_name = "breast_tcga"
random_state=123
block_num = 0
test_size = 0.25
# --------------------
# IMPLEMENTATION
# --------------------
raw_mdata = mudata.read(mu_path)
# Use a copy here to avoid mixing up stuff
mdata = raw_mdata.copy()

/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


In [3]:
def get_view_list(data_folder):
    # First list all *_featname.csv in current data folder
    pattern = "_featname.csv"
    csvs = [csv for csv in os.listdir(data_folder) if pattern in csv]
    # Remove the pattern, so left should be block name
    view_list = [csv.replace(pattern, "") for csv in csvs]
    return view_list


In [4]:
# Use this script as temp fix, need to update the pkg instead
import os
from sklearn.metrics import f1_score
import copy
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
# Mogonet pkg
from mogonet.train_mogonet import gen_trte_adj_mat
from mogonet.prepare_trte_data import prepare_trte_data
from mogonet.models import init_model_dict

# Othe functions to run

def test_epoch(data_list, adj_list, te_idx, model_dict):
    for m in model_dict:
        model_dict[m].eval()
    num_view = len(data_list)
    ci_list = []
    for i in range(num_view):
        ci_list.append(model_dict["C{:}".format(i+1)](model_dict["E{:}".format(i+1)](data_list[i],adj_list[i])))
    if num_view >= 2:
        c = model_dict["C"](ci_list)
    else:
        c = ci_list[0]
    c = c[te_idx,:]
    prob = F.softmax(c, dim=1).data.cpu().numpy()

    return prob

In [5]:
def cal_feat_imp(data_folder, view_list, num_class, he_base_dim = 100, adj_parameter = 10):
    num_view = len(view_list)
    dim_hvcdn = pow(num_class,num_view)
    # Fix later ------------------
    adj_parameter = adj_parameter
    dim_he_list = [he_base_dim] * num_view
    # Fix later -----------------
    data_tr_list, data_trte_list, trte_idx, labels_trte = prepare_trte_data(data_folder, view_list)
    adj_tr_list, adj_te_list = gen_trte_adj_mat(data_tr_list, data_trte_list, trte_idx, adj_parameter)
    featname_list = []
    for v in view_list:
        df = pd.read_csv(os.path.join(data_folder, str(v)+"_featname.csv"), header=None)
        featname_list.append(df.values.flatten())
    
    dim_list = [x.shape[1] for x in data_tr_list]
    model_dict = init_model_dict(num_view, num_class, dim_list, dim_he_list, dim_hvcdn)
    cuda = True if torch.cuda.is_available() else False
    for m in model_dict:
        if cuda:
            model_dict[m].cuda()
    # This bit is required, for the line:
    # feat_imp['imp'][j] = (f1-f1_tmp)*dim_list[i]
    te_prob = test_epoch(data_trte_list, adj_te_list, trte_idx["te"], model_dict)
    if num_class == 2:
        f1 = f1_score(labels_trte[trte_idx["te"]], te_prob.argmax(1))
    else:
        f1 = f1_score(labels_trte[trte_idx["te"]], te_prob.argmax(1), average='macro')
    
    feat_imp_list = []
    for i in range(len(featname_list)):
        feat_imp = {"feat_name":featname_list[i]}
        feat_imp['imp'] = np.zeros(dim_list[i])
        for j in range(dim_list[i]):
            feat_tr = data_tr_list[i][:,j].clone()
            feat_trte = data_trte_list[i][:,j].clone()
            data_tr_list[i][:,j] = 0
            data_trte_list[i][:,j] = 0
            adj_tr_list, adj_te_list = gen_trte_adj_mat(data_tr_list, data_trte_list, trte_idx, adj_parameter)
            te_prob = test_epoch(data_trte_list, adj_te_list, trte_idx["te"], model_dict)
            if num_class == 2:
                f1_tmp = f1_score(labels_trte[trte_idx["te"]], te_prob.argmax(1))
            else:
                f1_tmp = f1_score(labels_trte[trte_idx["te"]], te_prob.argmax(1), average='macro')
            feat_imp['imp'][j] = (f1-f1_tmp)*dim_list[i]
            data_tr_list[i][:,j] = feat_tr.clone()
            data_trte_list[i][:,j] = feat_trte.clone()
        feat_imp_list.append(pd.DataFrame(data=feat_imp))
    return feat_imp_list


In [22]:
d = {"x1": [1,5,2], "x3": [2,4,8]}
d2 = {"x1": [-8,-5,2], "x3": [12,0 , 1]}
a_df = pd.DataFrame(d)
b_df = pd.DataFrame(d2)

In [30]:
def summarize_imp_feat(featimp_list_list, dataset_name, topn=30, method='mogonet'):
    num_rep = len(featimp_list_list)
    num_view = len(featimp_list_list[0])
    df_tmp_list = []
    for v in range(num_view):
        df_tmp = copy.deepcopy(featimp_list_list[0][v])
        #df_tmp['omics'] = np.ones(df_tmp.shape[0], dtype=int)*v
        df_tmp['omics'] = view_list[v]
        df_tmp_list.append(df_tmp.copy(deep=True))
    df_featimp = pd.concat(df_tmp_list).copy(deep=True)
    for r in range(1,num_rep):
        for v in range(num_view):
            df_tmp = copy.deepcopy(featimp_list_list[r][v])
            #df_tmp['omics'] = np.ones(df_tmp.shape[0], dtype=int)*v
            df_tmp['omics'] = view_list[v]
            # THIS IS ORIGINALLY BUGGY CODE, pandas.DataFrame.append is deprecated now
            #df_featimp = df_featimp.append(df_tmp.copy(deep=True), ignore_index=True)
            df_featimp = pd.concat([df_featimp, df_tmp.copy(deep=True)])
    df_featimp_top = df_featimp.groupby(['feat_name', 'omics'])['imp'].sum()
    df_featimp_top = df_featimp_top.reset_index()
    df_featimp_top = df_featimp_top.sort_values(by='imp',ascending=False)
    # Add more metadata into it for dowsntream usage
    df_featimp_top["method"] = method
    df_featimp_top["dataset_name"] = dataset_name
    # Rename some columns
    df_featimp_top = df_featimp_top.rename(
      columns={
      'feat_name': 'feat', 
      'omics': 'view'
      })
    # And select everything else except this imp column
    df_featimp_top = df_featimp_top.drop(columns=['imp'])
    # Try to get those top n
    df_featimp_top = df_featimp_top.iloc[:topn]
    return df_featimp_top

In [7]:
outdir = prepare_mogonet_feat_select(mdata, dataset_name, random_state=random_state, 
                                    block_num=block_num, test_size=test_size)
data_folder = outdir
print("Data folder is", data_folder)
# TODO: Then need to run stuff (very ugly code....)
view_list = [*mdata.mod]

Saving mogonet inputs
Saved
Data folder is breast_tcga-random_state_123


In [8]:
import numpy as np
import copy
import pandas as pd

In [31]:
def main(mu_path, dataset_name, n_percent, random_state=123, block_num=0, test_size=0.25, reps=5, num_class=2):
    # ---------------
    # PARAMS
    # ---------------
    method = "mogonet"
    # --------------------
    # IMPLEMENTATION
    # --------------------
    raw_mdata = mudata.read(mu_path)
    # Use a copy here to avoid mixing up stuff
    mdata = raw_mdata.copy()
    view_list = [*mdata.mod]
    # Get top n from mdata
    # TODO: ...
    # It still needs to be splitted as specified format of mogonet

    # Allocate empty list to store result of different rep
    featimp_list_list = []
    for rep in range(reps):
        # Use a different random state to mimic "new splitting" for different repetitions
        new_random_state = random_state + rep
        data_folder = prepare_mogonet_feat_select(mdata, dataset_name, random_state=new_random_state, 
                                        block_num=block_num, test_size=test_size)
        print("Data folder is", data_folder)
        # =====================================================================================
        # This is the list of feature importance in single rep
        # he_base_dim is the dim of each he per omic, more like hidden layers?
        # adj_parameter needs to be tuned?
        featimp_list = cal_feat_imp(data_folder=data_folder, view_list=view_list, 
                                    num_class=num_class, he_base_dim = 100, adj_parameter = 10)
        # Add to the earlier allocated list
        featimp_list_list.append(copy.deepcopy(featimp_list))
    # Need to calculate the top N from the N percent
    topn = calculate_top_n(mdata=mdata, n_percent=n_percent)
    feats_df = summarize_imp_feat(featimp_list_list=featimp_list_list, dataset_name=dataset_name, topn=topn)
    filename = f"{method}-{dataset_name}_features_selected.csv"
    # And write it to file
    feats_df.to_csv(filename, index=False)
    return(feats_df)

In [38]:
df

,feat,view,method,dataset_name
185,MED13L,mrna,mogonet,breast_tcga
168,KIF13B,mrna,mogonet,breast_tcga
72,COL15A1,mrna,mogonet,breast_tcga
200,NDRG2,mrna,mogonet,breast_tcga
305,TTC39A,mrna,mogonet,breast_tcga
...,...,...,...,...
265,SGPP1,mrna,mogonet,breast_tcga
206,NRARP,mrna,mogonet,breast_tcga
209,OCLN,mrna,mogonet,breast_tcga
107,EIF2AK2,mrna,mogonet,breast_tcga


In [32]:

# Execute the main function
df = main(mu_path=mu_path, 
dataset_name=dataset_name,
n_percent=int(15)
)

/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Saving mogonet inputs
Saved
Data folder is breast_tcga-random_state_123
Saving mogonet inputs
Saved
Data folder is breast_tcga-random_state_124
Saving mogonet inputs
Saved
Data folder is breast_tcga-random_state_125
Saving mogonet inputs
Saved
Data folder is breast_tcga-random_state_126
Saving mogonet inputs
Saved
Data folder is breast_tcga-random_state_127
